## Imports

In [ ]:
import numpy as np
import pandas as pd
import ursu as ur

In [ ]:
data_ursu = pd.read_parquet("Ursu.parquet")

##### Setup

In [ ]:
n_sessions = 50
top_k = 7
R = 10_000
seed = 123

##### Build sessions

In [ ]:
sessions = ur.build_sessions(
    data_ursu,
    n_sessions=n_sessions,
    top_k=top_k,
    seed=seed
)

##### GMM estimation

In [ ]:
moments_msw = ["choice", "one_search", "two_search"]
moments_sp = ["choice", "one_search", "two_search", "searched_product"]


theta0 = np.array([-2.5, -1.5, 1.5, 1, 0.5], dtype=float)

In [ ]:
res_msw = ur.estimate_ursu_gmm(
    sessions=sessions,
    theta0=theta0,
    selected_moments=moments_msw,
    R=R
)

In [ ]:
res_sp = ur.estimate_ursu_gmm(
    sessions=sessions,
    theta0=theta0,
    selected_moments=moments_sp,
    R=R
)

##### Comparison

In [ ]:
parameter_names = [
    "beta_const",
    "beta_price",
    "beta_star",
    "lambda_const",
    "lambda_pos"
]

estimates = pd.DataFrame(
    {
        "MSW": res_msw.x,
        "MSW_SP": res_sp.x,
        "Difference": res_sp.x - res_msw.x
    },
    index=parameter_names
)

estimates = estimates.round(4)
estimates

In [ ]:
# =====================================================
# Prepare filtered sample
# =====================================================

df_prepared = ur.prepare_ursu_data(data_ursu)

session_ids = ur.select_ursu_sessions(
    df_prepared,
    n_sessions=50,
    top_k=7,
    seed=123
)

df_filtered = df_prepared[df_prepared["session_id"].isin(session_ids)].copy()

# =====================================================
# Session-level statistics
# =====================================================

session_stats = []

for session_id, df_s in df_filtered.groupby("session_id"):

    n_clicks = df_s["searched"].sum()
    booked = int(df_s["booked"].sum() > 0)

    session_stats.append({
        "session_id": session_id,
        "n_clicks": n_clicks,
        "booked": booked,
        "outside_option": 1 - booked
    })

stats = pd.DataFrame(session_stats)

# =====================================================
# Summary statistics
# =====================================================

summary = pd.DataFrame({
    "Statistic": [
        "Number of sessions",
        "Share zero clicks",
        "Share outside option",
        "Share booking",
        "Share exactly one click",
        "Share more than one click",
        "Average number of clicks"
    ],
    "Value": [
        len(stats),
        (stats["n_clicks"] == 0).mean(),
        stats["outside_option"].mean(),
        stats["booked"].mean(),
        (stats["n_clicks"] == 1).mean(),
        (stats["n_clicks"] > 1).mean(),
        stats["n_clicks"].mean()
    ]
})

summary